## LLM as a judge with a local llama.cpp server

LLMs are not only used for conversational AI but also to evaluate the conversations. For example, Bavareso et al (2025) tested 11 LLMs against JUDGE-BENCH, a collection of 20 NLLP datasets with human annotations. LLMs are not specififically designed or trained to judge conversations but have some understanding of conversations and should be able to reflect on it. 

In this notebook, we will prompt an LLM to score agent responses for coherence and also to explain how it came to this score. We will use a local LLM in GGUF format that is served by a [llama.cpp](https://github.com/ggml-org/llama.cpp) server. The llama.cpp server exposes an OpenAI-compatible API, so we can call it with the standard ```openai``` Python client. Obviously, any other platform (OpenAI, Google, ollama) can be used for the same purpose by changing the client. 

We will load the conversations from the EMISSOR representations of conversations by loading pairs of user input and agent responses.

## Reference
Anna Bavaresco, Raffaella Bernardi, Leonardo Bertolazzi, Desmond Elliott, Raquel Fernández, Albert Gatt, Esam Ghaleb, Mario Giulianelli, Michael Hanna, Alexander Koller, Andre Martins, Philipp Mondorf, Vera Neplenbroek, Sandro Pezzelle, Barbara Plank, David Schlangen, Alessandro Suglia, Aditya K Surikuchi, Ece Takmaz, and Alberto Testoni. 2025. LLMs instead of Human Judges? A Large Scale Empirical Study across 20 NLP Evaluation Tasks. In Proceedings of the 63rd Annual Meeting of the Association for Computational Linguistics (Volume 2: Short Papers), pages 238–255, Vienna, Austria. Association for Computational Linguistics.

## Prerequisites
This notebook requires packages installed for running an ```openai``` client and loading the EMISSOR scenarios. We also need ```pandas``` for saving the results in a spreadsheet.
These packages can be installed through ```pip```:

```
!pip install openai
!pip install emissor
!pip install pandas
```

You also need a local installation of [llama.cpp](https://github.com/ggml-org/llama.cpp) (e.g. ```brew install llama.cpp``` on macOS) and a model in GGUF format, for example from [huggingface.co](https://huggingface.co/models?library=gguf).

In [12]:
import os
import re
from openai import OpenAI

## Starting the llama.cpp server

Start the llama.cpp server in a terminal with your local GGUF model before running the rest of this notebook:

```bash
llama-server -m /path/to/your/model.gguf --port 8080 -c 4096
```

Alternatively, llama.cpp can download a GGUF model from huggingface directly:

```bash
llama-server -hf Qwen/Qwen2.5-7B-Instruct-GGUF:Q4_K_M --port 8080 -c 4096
```

The server then provides an OpenAI-compatible API at ```http://localhost:8080/v1```. No API key is needed, but the ```openai``` client requires a value, so any string will do.

## Defining an OpenAI client for the llama.cpp server and the model

In [13]:
LLAMA_CPP_URL = "http://localhost:8080/v1"

llama_client = OpenAI(base_url=LLAMA_CPP_URL, api_key="sk-no-key-required")

# llama.cpp serves the model it was started with, the name is only used for reporting
model = llama_client.models.list().data[0].id
print('Model served by llama.cpp:', model)

Model served by llama.cpp: Qwen/Qwen2.5-7B-Instruct-GGUF:Q4_K_M


## Prompting for coherence

The next two functions define the prompt for evaluating the coherence of the agent responses to the user. The prompt is adapted to take into account that the Leolani agent is proactive in driving the conversation and can ask questions to the user as well.

In [14]:
### differentiates between the role of the user and the assistant
def get_judge_prompt():
    instruction = """
                    ### Role Assignment
                    You are a Coherence Evaluation Judge.
                    Your job is to evaluate how coherent the **assistant’s response** is 
                    with respect to the **user’s input**.
                    The **assistant’s response** may be a follow up question to a 
                    statement from the **user**,  a related statement by the **agent** or the answer 
                    from the **agent** to a question of the **user**.
                    
                    ### Task Definition
                    You must:
                    1. Assign a **coherence score** from **0.0 to 1.0**
                    2. Provide a **short explanation** (maximum 2 sentences)
                    
                    ### Output Format (STRICT)
                    Return ONLY:
                    
                    <JSON>
                    {
                      "coherence_score": float between 0.0 and 1.0,
                      "explanation": "brief rationale"
                    }
                    </JSON>
                    """

    system_prompt = {
        "role": "system",
        "content": instruction
    }
    return system_prompt

# treate the user and assistant as equal interlocutors
def get_judge_prompt_interlocutor():
    instruction = """
                    ### Role Assignment
                    You are a Coherence Evaluation Judge.
                    You will receive a pair of utterances from a **user** and an **assistant** as 
                    interlocutors or the other way around.
                    Your job is to evaluate how coherent an **interlocutor’s response** is with 
                    respect to the previous input of the other **interlocutor**.
                    The **interlocutor’s response** can be a follow up question to a previous statement,  
                    the **interlocutor’s response** can be a related statement 
                    or the **interlocutor’s response** can be an answer to a question of the **user**.
                    Always consider the coherence of the last utterance given the first 
                    utterance in a pair of utterances.
                    
                    ### Task Definition
                    You must:
                    1. Assign a **coherence score** from **0.0 to 1.0**
                    2. Provide a **short explanation** (maximum 2 sentences)
                    
                    ### Output Format (STRICT)
                    Return ONLY:
                    
                    <JSON>
                    {
                      "coherence_score": float between 0.0 and 1.0,
                      "explanation": "brief rationale"
                    }
                    </JSON>
                    """

    system_prompt = {
        "role": "system",
        "content": instruction
    }
    return system_prompt
    
def query_llm_as_a_judge_llama_cpp(messages, client, model):
    #system_prompt = get_judge_prompt()
    system_prompt = get_judge_prompt_interlocutor()
    # llama.cpp treats a final "assistant" message as the start of its own answer and continues it
    # (or rejects two assistant messages in a row), so we pass the pair as text in a single user message
    pair_text = ""
    for index, message in enumerate(messages):
        pair_text += f"Utterance {index + 1} ({message['role']}): {message['content']}\n"
    messages_judge = [system_prompt, {"role": "user", "content": pair_text.strip()}]
    try:
        completion = client.chat.completions.create(model=model,
                                                    messages=messages_judge,
                                                    temperature=0.0,
                                                    response_format={"type": "json_object"})
        response = completion.choices[0].message.content
    except Exception as e:
        return {"coherence_score": None, "explanation": f"Error calling llama.cpp server: {str(e)}"}
    # Look for patterns like "coherence_score": 0.9 or similar
    try:
        score_pattern = r'"coherence_score":\s*([\d.]+)'
        explanation_pattern = r'"explanation":\s*"([^"]+)"'

        score_match = re.search(score_pattern, response)
        explanation_match = re.search(explanation_pattern, response)

        if score_match and explanation_match:
            coherence_score = float(score_match.group(1))
            explanation = explanation_match.group(1)
            return {"coherence_score": coherence_score, "explanation": explanation}
        else:
            return {"coherence_score": None, "explanation": "Could not extract score and explanation from response"}
    except Exception as e:
        return {"coherence_score": None, "explanation": f"Error processing response: {str(e)}"}

## Obtaining the conversations from EMISSOR

The next two functions get the text signals from a conversation captured in an EMISSOR scenario and get the speaker of a text signal that represents a turn.

In [ ]:
import sys
from pathlib import Path

# Jupyter runs the notebook in its own folder. We add the root of the project to the path,
# so that the modules can be imported from src.cltl.emissor_evaluation
PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "cltl").is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [15]:

from src.cltl.emissor_evaluation.emissor import emissor_util as util

In [16]:
EMISSOR="../../../../emissor"
SCENARIO="2ad0c8b9-4e62-4f11-809c-57bf76487f95"

EMISSOR="/Users/piek/Desktop/Leolani/cltl-apps/cltl-apps-restructure_f_server/storage/emissor"
SCENARIO="5a55e02f-e2be-4dce-a9b6-bb3c6dba1480"

## Scoring conversational pairs for coherence

The next code shows how the conversation from EMISSOR is processed as pairs of user input and agent responses and each of these is passed on the to LLM for assessing the coherence. It extracts a score for each pair and averages the score over all pairs.

In [17]:
text_signals = util.get_text_signals_from_a_scenario(EMISSOR, SCENARIO)
coherence_results = []
previous_turn = {"role": "user", "content": ""}
for text_signal in text_signals:
    speaker = util.get_speaker_from_text_signal(text_signal) # SPEAKER or agent
    role = "assistant"
    if not speaker=="LEOLANI":
        role = "user"
    turn = {"role":role, "content": text_signal.text}
    pair = [previous_turn, turn]
    coherence = query_llm_as_a_judge_llama_cpp(pair, llama_client, model)
    coherence_score = coherence.get("coherence_score")
    explanation = coherence.get("explanation", "")
    print(previous_turn, turn)
    print(coherence)
    row = {"Turn": text_signal.id, "Speaker": speaker, "Response": text_signal.text, "llm_coherence": coherence_score, "llm_explanation": explanation}
    coherence_results.append(row)
    previous_turn = turn

{'role': 'user', 'content': ''} {'role': 'assistant', 'content': "What's up? Do you want to talk to me?"}
{'coherence_score': 0.5, 'explanation': "The assistant's response is somewhat related to the user's input, as it could be a follow-up to an implied question or statement. However, it lacks direct coherence and relevance, as it doesn't address any specific point or question from the user."}
{'role': 'assistant', 'content': "What's up? Do you want to talk to me?"} {'role': 'user', 'content': 'yes'}
{'coherence_score': 1.0, 'explanation': "The user's response directly answers the assistant's question, showing coherence."}
{'role': 'user', 'content': 'yes'} {'role': 'user', 'content': 'look'}
{'coherence_score': 0.0, 'explanation': "The second utterance does not coherently follow the first 'yes', as it introduces a new action without context."}
{'role': 'user', 'content': 'look'} {'role': 'assistant', 'content': 'What’s been going on with your body lately—have you noticed any changes i

In [18]:
speaker_coherence = []
agent_coherence = []
for result in coherence_results:
    print(result["Turn"], ":", result["llm_explanation"])
    if result["llm_coherence"] is None:
        continue
    if result["Speaker"]=="LEOLANI":
        agent_coherence.append(result["llm_coherence"])
    else:
        speaker_coherence.append(result["llm_coherence"])

average_speaker_coherence = sum(speaker_coherence)/len(speaker_coherence)
average_agent_coherence = sum(agent_coherence)/len(agent_coherence)

print('average_speaker_coherence',average_speaker_coherence)
print(speaker_coherence)
print('average_agent_coherence',average_agent_coherence)
print(agent_coherence)

f84ea793-dd7f-4f69-8280-3b3b877351e0 : The assistant's response is somewhat related to the user's input, as it could be a follow-up to an implied question or statement. However, it lacks direct coherence and relevance, as it doesn't address any specific point or question from the user.
f660f8f5-2713-4d86-bfbc-0c07c8cafc95 : The user's response directly answers the assistant's question, showing coherence.
e13f51d5-f2cd-45ea-bab8-4d7bca228f33 : The second utterance does not coherently follow the first 'yes', as it introduces a new action without context.
f5d92bd0-dcb3-48c6-834e-45907f92d67e : The user's 'look' is a single word with no context, and the assistant's response is a detailed question about physical changes, which is not coherent given the minimal input.
0c012f51-d4da-451d-9cce-fe771ae6f359 : The second response does not follow up on the first question about physical changes, but instead shifts to a new topic about mood and headspace, which is not coherent with the previous utt

Running the code again with ```temperature=0.0``` should give you (nearly) the same results. If you increase the temperature, it is wise to run the code several times and average over the different runs.

## Saving the coherence score to a CSV file

We save the conversation with the coherence score to a CSV file to compare it against other evaluations and human scores.

In [19]:
import pandas as pd

In [20]:
df = pd.DataFrame(coherence_results)

evaluation_folder = os.path.join(EMISSOR, SCENARIO, 'evaluation')
if not os.path.exists(evaluation_folder):
    os.mkdir(evaluation_folder)
file_name =  SCENARIO+"_llm_judge_llama_cpp_evaluation.csv"
file = os.path.join(evaluation_folder, file_name)
df.to_csv(file, sep=";", index=False)

## End of notebook